### Citation Dynamics
We further explor how citations differ **over time** between sharing and non-sharing articles; and whether different types of data sharing are associated with different citation dynamics.<br>
To do so, we calculate the number of citations in each calendar year post-publication for each article (ignoring "year 0" due to partial year effects). We then average these yearly citation counts across articles within each sharing category, and examine the differences.

In [1]:
import numpy as np
import pandas as pd
import pingouin as pg
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from helpers.config import *
from helpers import dataset
from helpers.plotting import save_figure

pio.renderers.default = "browser"

In [2]:
combined, FEATURES_DF = dataset.load_or_build()

Loaded cached dataset from C:\Users\nirjo\Documents\University\PhD\Projects\eye_movement_data_sharing\.claude\worktrees\eye-movement-analysis-review-22482b\data_store (234 rows)


In [3]:
def citations_per_year(
        articles: pd.DataFrame, grouping_col: str, cumulative: bool = True
) -> pd.DataFrame:
    """Citations per year since publication, indexed by (article, group).

    The reshape from absolute calendar years to years-since-publication is done by
    `dataset.citations_since_publication()`. This wrapper only applies the two analysis choices
    belonging to this notebook, which that helper deliberately leaves to its callers: drop the
    publication year itself (`year_0`, partial for every article), and drop any calendar year that
    has not fully elapsed for the whole sample.

    The incomplete-year cutoff comes from `dataset.citations_through_shared_year()` rather than a
    hard-coded year, so it tracks the data instead of going stale at the next re-fetch.

    :param articles: frame carrying the `Citations20XX` columns, `PublicationYear`, `LastUpdate`
        and `grouping_col`.
    :param grouping_col: column to append as the second index level, e.g. `is_sharing_data`.
    :param cumulative: True for cumulative citations (stock), False for per-year counts (flow).
    :return: frame indexed by (article, `grouping_col`), one column per year offset.
    """
    _, shared_year = dataset.citations_through_shared_year(articles)
    incomplete = [
        col for col in articles.columns
        if col.startswith("Citations") and col.removeprefix("Citations").isdigit()
        and int(col.removeprefix("Citations")) > shared_year
    ]

    per_year = dataset.citations_since_publication(articles.drop(columns=incomplete))
    per_year = per_year.drop(columns="year_0")
    per_year.columns = [int(col.removeprefix("year_")) for col in per_year.columns]
    if cumulative:
        per_year = per_year.cumsum(axis=1)
    return per_year.set_index(articles[grouping_col], append=True)

#### (1) Sharing vs. Non-Sharing Articles
We compare the cumulative citation counts per year between articles that share data vs. those that do not.

In [4]:
GROUPING_COL = "is_sharing_data"

# cumulative shows the total citations up to (and including) each year (stock)
# non-cumulative shows the "velocity" of citations per year (flow)
cit_dynamics_sharing = citations_per_year(
    combined, grouping_col=GROUPING_COL,
    cumulative=True
)
# drop years with fewer than 20 articles combined
cit_dynamics_sharing = cit_dynamics_sharing.loc[:, cit_dynamics_sharing.count(axis=0) >= 20]

year_stats = dict()
for year in cit_dynamics_sharing.columns:
    non_sharing_vals = cit_dynamics_sharing.xs(False, level=GROUPING_COL)[year].dropna()
    non_sharing_is_normal = stats.shapiro(non_sharing_vals).pvalue > 0.05
    sharing_vals = cit_dynamics_sharing.xs(True, level=GROUPING_COL)[year].dropna()
    sharing_is_normal = stats.shapiro(sharing_vals).pvalue > 0.05
    n1, n2 = len(non_sharing_vals), len(sharing_vals)
    yearly_ttest = pg.ttest(non_sharing_vals, sharing_vals, paired=False, alternative="less", correction=True)
    yearly_mwtest = pg.mwu(non_sharing_vals, sharing_vals, alternative="less")
    U_nonshare = yearly_mwtest.loc["MWU", "U_val"]
    U_share = n1 * n2 - U_nonshare
    rank_biserial_corr = (U_share - U_nonshare) / (n1 * n2)
    cles = U_share / (n1 * n2)
    year_stats[year] = {
        "non_sharing_count": n1,
        "non_sharing_is_normal": non_sharing_is_normal,
        "non_sharing_median": non_sharing_vals.median(),
        "non_sharing_mean": non_sharing_vals.mean(),
        "non_sharing_sem": non_sharing_vals.sem(),

        "sharing_count": n2,
        "sharing_is_normal": sharing_is_normal,
        "sharing_median": sharing_vals.median(),
        "sharing_mean": sharing_vals.mean(),
        "sharing_sem": sharing_vals.sem(),

        "t_stat": yearly_ttest.loc["T_test", "T"],
        "t_dof": yearly_ttest.loc["T_test", "dof"],
        "t_p_raw": yearly_ttest.loc["T_test", "p_val"],
        "t_cohen_d": yearly_ttest.loc["T_test", "cohen_d"],

        "U_nonshare": U_nonshare,
        "U_share": U_share,
        "mw_p_raw": yearly_mwtest.loc["MWU", "p_val"],
        "r_rb": abs(yearly_mwtest.loc["MWU", "RBC"]),
        "cles": yearly_mwtest.loc["MWU", "CLES"],
    }

year_stats = pd.DataFrame.from_dict(year_stats, orient='index').reset_index().rename(columns={"index": "year"})
t_reject, t_p_corrected, _, _ = multipletests(year_stats['t_p_raw'], alpha=0.05, method='fdr_bh')
mw_reject, mw_p_corrected, _, _ = multipletests(year_stats['mw_p_raw'], alpha=0.05, method='fdr_bh')
year_stats['t_p_corrected'] = t_p_corrected
year_stats['t_is_significant'] = t_reject
year_stats['mw_p_corrected'] = mw_p_corrected
year_stats['mw_is_significant'] = mw_reject
column_order = [
    "year",
    "non_sharing_is_normal", "sharing_is_normal",
    "t_is_significant", "t_p_corrected",
    "mw_is_significant", "mw_p_corrected",
    "t_stat", "t_dof", "t_p_raw", "t_cohen_d",
    "U_share", "U_nonshare", "r_rb", "cles",
    "non_sharing_count", "non_sharing_mean", "non_sharing_sem", "non_sharing_median",
    "sharing_count", "sharing_mean", "sharing_sem", "sharing_median",
]
year_stats = year_stats[column_order]

year_stats

,year,non_sharing_is_normal,sharing_is_normal,t_is_significant,t_p_corrected,mw_is_significant,mw_p_corrected,t_stat,t_dof,t_p_raw,...,r_rb,cles,non_sharing_count,non_sharing_mean,non_sharing_sem,non_sharing_median,sharing_count,sharing_mean,sharing_sem,sharing_median
0,1,False,False,True,0.042127,True,0.025049,-2.186395,105.167488,0.015502,...,0.214692,0.607346,154,2.422078,0.209442,2.0,80,3.650000,0.521105,2.0
1,2,False,False,True,0.042127,True,0.038688,-2.126494,134.212422,0.017646,...,0.168344,0.584172,154,5.883117,0.467058,4.5,80,7.850000,0.798357,6.0
2,3,False,False,True,0.042127,True,0.038688,-2.140418,139.002827,0.017033,...,0.164692,0.582346,154,9.207792,0.686284,7.0,80,12.025000,1.123114,9.0
3,4,False,False,True,0.042127,True,0.038688,-2.053159,125.961483,0.021064,...,0.183099,0.591549,141,12.539007,0.981930,9.0,71,16.338028,1.568290,13.0
4,5,False,False,False,0.076211,False,0.077369,-1.571919,68.651617,0.060284,...,0.150164,0.575082,119,15.117647,1.227746,11.0,46,19.434783,2.456708,16.0
5,6,False,False,False,0.076211,False,0.077369,-1.551163,43.484219,0.064056,...,0.181726,0.590863,92,18.391304,1.652429,14.0,32,24.781250,3.773511,19.5
6,7,False,False,False,0.076211,False,0.065758,-1.554672,23.535548,0.066685,...,0.263968,0.631984,65,22.046154,2.303784,18.0,19,32.052632,6.009969,28.0
7,8,False,True,False,0.187930,False,0.167059,-0.908529,17.617198,0.187930,...,0.188596,0.594298,38,25.026316,3.333627,21.0,12,31.500000,6.297546,27.5


In [5]:
LINE_ANNOTATION_X = 3

cit_dynamics_sharing_fig = go.Figure()
for is_share in cit_dynamics_sharing.index.get_level_values(GROUPING_COL).unique().sort_values():
    subset = cit_dynamics_sharing.xs(is_share, level=GROUPING_COL)
    count_per_year = subset.count(axis=0)
    mean_per_year = subset.mean(axis=0)
    sem_per_year = subset.sem(axis=0)
    mean_citations_per_year = subset.mean(axis=0)
    label = "Sharing" if is_share else "Not Sharing"
    color = CLASS_COLORS[label.upper()]
    cit_dynamics_sharing_fig.add_trace(go.Scatter(
        name=label, legendgroup=label,
        x=mean_per_year.index,
        y=mean_per_year.values,
        error_y=dict(type='data', array=sem_per_year.values, width=5, visible=True,),
        marker=dict(
            color=color,
            # size=np.sqrt(count_per_year.values),
        ),
        mode='lines+markers',
    ))
    # annotate the lines
    if is_share:
        ann_y = mean_per_year[LINE_ANNOTATION_X] + 1.5 * sem_per_year[LINE_ANNOTATION_X]
        xanchor = "right"
        yanchor = "bottom"
    else:
        ann_y = mean_per_year[LINE_ANNOTATION_X] - 1.5 * sem_per_year[LINE_ANNOTATION_X]
        xanchor = "left"
        yanchor = "top"
    cit_dynamics_sharing_fig.add_annotation(
        x=LINE_ANNOTATION_X, xanchor=xanchor, xref="x",
        y=ann_y, yanchor=yanchor, yref="y",
        text=label, font={**AXIS_TITLE_FONT, "color": color},
        showarrow=False,
    )

# add significance asterisks
SIGNIFICANCE_ANNOTATION_Y = 40
for _, row in year_stats.iterrows():
    if row["mw_p_corrected"] >= 0.1:
        continue
    fontsize = 22
    yanchor = "middle"
    if row["mw_p_corrected"] < 0.001:
        ann_text = "<b>***</b>"
    elif row["mw_p_corrected"] < 0.01:
        ann_text = "<b>**</b>"
    elif row["mw_p_corrected"] < 0.05:
        ann_text = "<b>*</b>"
    else:
        ann_text = "\u2020"     # dagger for p < 0.
        fontsize = 16
        yanchor = "bottom"
    cit_dynamics_sharing_fig.add_annotation(
        x=row["year"], xanchor="center", xref="x",
        y=SIGNIFICANCE_ANNOTATION_Y, yanchor=yanchor, yref="y",
        text=ann_text, font={**TITLE_FONT, "size": fontsize},
        showarrow=False,
    )

cit_dynamics_sharing_fig.update_layout(
    width=800, height=400,
    title=dict(
        text="Citation Dynamics: Sharing vs. Non-Sharing Articles",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top"
    ),
    xaxis=dict(
        title=dict(text="Years Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Mean Cumulative Citation per Year", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        showgrid=True, gridcolor='lightgrey', gridwidth=1.5,
        zeroline=False,
    ),
    legend=dict(
        title=dict(text="Data Sharing", font=AXIS_TITLE_FONT),
        font=LEGEND_FONT,
        x=1.0, xanchor="right", y=0.4, yanchor="top",
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black', borderwidth=1,
        visible=False,
    ),
    margin=dict(t=75, b=50, l=50, r=25, pad=0),
    template="plotly_white"
)
cit_dynamics_sharing_fig.show()

In [6]:
# flip to True to export this figure into `output/`
if False:
    save_figure(cit_dynamics_sharing_fig, "citation_dynamics_sharing_vs_nonsharing.png", width=800, height=400)

##### Source of the Difference
So we found a difference in yearly cumulative citations between sharing and non-sharing articles.<br>
But where does this difference come from? Is it because the sharing articles get cited more frequently in each year (i.e., higher "velocity" of citations)? Or is it because they got an initian citation boost which then accumulates over time (i.e., higher "stock" of citations)?<br>
To answer this question, we repeat the above analysis but now looking at **non-cumulative** citation counts per year (i.e., the "velocity" of citations).

In [7]:
GROUPING_COL = "is_sharing_data"

# cumulative shows the total citations up to (and including) each year (stock)
# non-cumulative shows the "velocity" of citations per year (flow)
cit_dynamics_sharing = citations_per_year(
    combined, grouping_col=GROUPING_COL,
    cumulative=False
)

year_stats = dict()
for year in cit_dynamics_sharing.columns:
    non_sharing_vals = cit_dynamics_sharing.xs(False, level=GROUPING_COL)[year].dropna()
    sharing_vals = cit_dynamics_sharing.xs(True, level=GROUPING_COL)[year].dropna()
    n1, n2 = len(non_sharing_vals), len(sharing_vals)
    year_pair_test = stats.mannwhitneyu(non_sharing_vals, sharing_vals, alternative="less")
    U_nonshare = year_pair_test.statistic
    U_share = n1 * n2 - U_nonshare
    rank_biserial_corr = (U_share - U_nonshare) / (n1 * n2)
    cles = U_share / (n1 * n2)
    year_stats[year] = {
        "non_sharing_count": n1,
        "non_sharing_mean": non_sharing_vals.mean(),
        "non_sharing_sem": non_sharing_vals.sem(),
        "sharing_count": n2,
        "sharing_mean": sharing_vals.mean(),
        "sharing_sem": sharing_vals.sem(),
        "U_nonshare": U_nonshare,
        "p_raw": year_pair_test.pvalue,
        "r_rb": rank_biserial_corr,
        "cles": cles,
    }

year_stats = pd.DataFrame.from_dict(year_stats, orient='index').reset_index().rename(columns={"index": "year"})
reject, p_corrected, _, _ = multipletests(year_stats['p_raw'], alpha=0.05, method='fdr_bh')
year_stats['p_corrected'] = p_corrected
year_stats['is_significant'] = reject
column_order = [
    "year",
    "is_significant", "p_corrected", "U_nonshare", "r_rb", "cles",
    "non_sharing_count", "non_sharing_mean", "non_sharing_sem",
    "sharing_count", "sharing_mean", "sharing_sem",
    "p_raw"
]
year_stats = year_stats[column_order]

year_stats

,year,is_significant,p_corrected,U_nonshare,r_rb,cles,non_sharing_count,non_sharing_mean,non_sharing_sem,sharing_count,sharing_mean,sharing_sem,p_raw
0,1,True,0.028180,4837.5,0.214692,0.607346,154,2.422078,0.209442,80,3.650000,0.521105,0.003131
1,2,False,0.248747,5563.5,0.096834,0.548417,154,3.461039,0.291511,80,4.200000,0.456112,0.110554
2,3,False,0.248747,5468.5,0.112256,0.556128,154,3.324675,0.262354,80,4.175000,0.436026,0.077921
3,4,False,0.495414,4951.0,0.010888,0.505444,141,3.425532,0.331897,71,3.605634,0.455893,0.448522
4,5,False,0.495414,2677.5,0.021739,0.510870,119,3.016807,0.317550,46,3.217391,0.574761,0.413776
5,6,False,0.248747,1259.5,0.144361,0.572181,92,2.880435,0.385735,32,3.843750,0.800843,0.109784
6,7,False,0.495414,567.5,0.080972,0.540486,65,3.276923,0.422053,19,4.000000,1.102894,0.296125
7,8,False,0.495414,227.0,0.004386,0.502193,38,2.894737,0.469238,12,2.916667,0.820738,0.495414
8,9,False,0.495414,6.5,0.187500,0.593750,8,4.500000,1.812654,2,4.500000,0.500000,0.396386


### (2) Different Types of Data Sharing

In [8]:
GROUPING_COL = "data_sharing_class"

cit_dynamics_sharing_class = citations_per_year(
    combined.loc[combined["is_sharing_data"]], grouping_col=GROUPING_COL
)

year_stats = dict()
for year in cit_dynamics_sharing_class.columns:
    year_stats[year] = dict()
    class_values = []
    for share_class in cit_dynamics_sharing_class.index.get_level_values(GROUPING_COL).unique():
        subset = (cit_dynamics_sharing_class.xs(share_class, level=GROUPING_COL)[year]).dropna()
        if subset.empty:
            continue
        year_stats[year][f"{share_class}_count"] = subset.count()
        year_stats[year][f"{share_class}_mean"] = subset.mean()
        year_stats[year][f"{share_class}_sem"] = subset.sem()
        class_values.append(subset)
    sharegroup_test = stats.kruskal(*class_values)
    year_stats[year]["W"] = sharegroup_test.statistic
    year_stats[year]["p_raw"] = sharegroup_test.pvalue

year_stats = pd.DataFrame.from_dict(year_stats, orient='index')
reject, p_corrected, _, _ = multipletests(year_stats['p_raw'], alpha=0.05, method='fdr_bh')
year_stats['p_corrected'] = p_corrected
year_stats['is_significant'] = reject
column_order = [
    "is_significant", "p_corrected", "W", "p_raw",
    *[f"{share_class}_{stat}" for share_class in ["FIXATION", "TRIAL", "PARTICIPANT"] for stat in ["count", "mean", "sem"]]
]
year_stats = year_stats[column_order]
year_stats

,is_significant,p_corrected,W,p_raw,FIXATION_count,FIXATION_mean,FIXATION_sem,TRIAL_count,TRIAL_mean,TRIAL_sem,PARTICIPANT_count,PARTICIPANT_mean,PARTICIPANT_sem
1,False,0.475966,3.703398,0.156970,31,4.741935,1.198537,26.0,2.500000,0.493808,23,3.478261,0.543992
2,False,0.475966,2.734833,0.254764,31,9.225806,1.513885,26.0,6.576923,1.339544,23,7.434783,1.111965
3,False,0.475966,2.548764,0.279604,31,13.354839,1.877741,26.0,10.115385,1.992545,23,12.391304,1.967863
4,False,0.475966,2.394341,0.302048,28,18.142857,2.503286,24.0,14.208333,2.874790,19,16.368421,2.822436
5,False,0.481202,1.809868,0.404569,21,21.809524,3.586761,12.0,14.750000,4.407509,13,19.923077,5.152124
6,False,0.475966,2.332843,0.311480,15,28.666667,5.428028,6.0,14.500000,4.667262,11,25.090909,7.641340
7,False,0.481202,1.462937,0.481202,8,33.375000,7.690294,4.0,19.750000,6.289873,7,37.571429,13.569674
8,False,0.481202,1.488722,0.475038,7,39.857143,9.230974,2.0,23.500000,14.500000,3,17.333333,1.666667
9,False,0.475966,1.000000,0.317311,1,48.000000,NaN,NaN,NaN,NaN,1,23.000000,NaN


In [9]:
cit_dynamics_sharing_class_fig = go.Figure()
for share_class in cit_dynamics_sharing_class.index.get_level_values(GROUPING_COL).unique():
    subset = cit_dynamics_sharing_class.xs(share_class, level=GROUPING_COL)
    count_per_year = subset.count(axis=0)
    mean_per_year = subset.mean(axis=0)
    sem_per_year = subset.sem(axis=0)
    mean_citations_per_year = subset.mean(axis=0)
    cit_dynamics_sharing_class_fig.add_trace(go.Scatter(
        name=share_class, legendgroup=share_class,
        x=mean_per_year.index,
        y=mean_per_year.values,
        error_y=dict(type='data', array=sem_per_year.values, width=5, visible=True,),
        marker=dict(
            color=CLASS_COLORS[share_class],
            size=count_per_year.values
        ),
        mode='lines+markers',
    ))

cit_dynamics_sharing_class_fig.update_layout(
    width=800, height=400,
    title=dict(
        text="Citation Dynamics: Different Types of Data Sharing",
        font=TITLE_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top"
    ),
    xaxis=dict(
        title=dict(text="Years Since Publication", font=AXIS_TITLE_FONT, standoff=10),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    yaxis=dict(
        title=dict(text="Mean Cumulative Citation per Year", font=AXIS_TITLE_FONT, standoff=5),
        tickfont=AXIS_TICK_FONT,
        zeroline=False,
    ),
    legend=dict(
        title=dict(text="Data Sharing", font=AXIS_TITLE_FONT),
        font=LEGEND_FONT,
        x=0.5, xanchor="center", y=0.95, yanchor="top",
        bgcolor='rgba(255,255,255,0.5)',
        bordercolor='black', borderwidth=1,
    ),
    margin=dict(t=75, b=50, l=50, r=25, pad=0),
    template="plotly_white"
)
cit_dynamics_sharing_class_fig.show()

In [10]:
# flip to True to export this figure into `output/`
if False:
    save_figure(cit_dynamics_sharing_class_fig, "citation_dynamics_sharing_class.png", width=800, height=400)

---
### TODO: age- and venue-adjusted citation comparisons
(was cell 82)

1. build a feature matrix with citation counts per year-post-publication
2. run a mixed-effect model regressing `cumulative citations k-years post publication` on all covariates; use journal as a categorical rather than numeric predictor, and publication year as another categorical predictor (or random effect - need to check which is better)
3. bring back the old year-by-year comparison from the previous version; add a markdown explaining why that analysis is not included in the article, and that the new mixed-effect model is a better approach